# DATA-445: In-Class Assignment 15

Consider the `framingham.csv` data file. The dataset is available on the Kaggle website, and it is from an ongoing cardiovascular study on residents of the town of Framingham, Massachusetts. The classification goal is to predict whether the patient has 10-year risk of future coronary heart disease (CHD). The dataset provides the patients' information. It includes over 4,000 records and 15 attributes. Each attribute is a potential risk factor. There are both demographic, behavioral and medical risk factors.

**Demographic:**
- Sex: male or female (Nominal)
- Age: Age of the patient (Continuous - Although the recorded ages have been truncated to whole numbers, the concept of age is continuous)

**Behavioral:**
- Current Smoker: whether or not the patient is a current smoker (Nominal)
- Cigs Per Day: the number of cigarettes that the person smoked on average in one day (can be considered continuous as one can have any number of cigarettes, even half a cigarette)

**Medical (history):**
- BP Meds: whether or not the patient was on blood pressure medication (Nominal)
- Prevalent Stroke: whether or not the patient had previously had a stroke (Nominal)
- Prevalent Hyp: whether or not the patient was hypertensive (Nominal)
- Diabetes: whether or not the patient had diabetes (Nominal)

**Medical (current):**
- Tot Chol: total cholesterol level (Continuous)
- Sys BP: systolic blood pressure (Continuous)
- Dia BP: diastolic blood pressure (Continuous)
- BMI: Body Mass Index (Continuous)
- Heart Rate: heart rate (Continuous - In medical research, variables such as heart rate though in fact discrete, yet are considered continuous because of large number of possible values)
- Glucose: glucose level (Continuous)

**Predict variable (desired target):**
- 10 year risk of coronary heart disease CHD (binary: "1" means "Yes", "0" means "No")

In **Python**, answer the following:


## 1. (4 points)

Using the `pandas` library, read the csv data file and create a data-frame called `heart`. Remove the missing values.

In [1]:
# Your code here
import pandas as pd
import numpy as np

heart = pd.read_csv("framingham.csv")
heart = heart.dropna()
heart.head

<bound method NDFrame.head of       male  age  education  currentSmoker  cigsPerDay  BPMeds  \
0        1   39        4.0              0         0.0     0.0   
1        0   46        2.0              0         0.0     0.0   
2        1   48        1.0              1        20.0     0.0   
3        0   61        3.0              1        30.0     0.0   
4        0   46        3.0              1        23.0     0.0   
...    ...  ...        ...            ...         ...     ...   
4231     1   58        3.0              0         0.0     0.0   
4232     1   68        1.0              0         0.0     0.0   
4233     1   50        1.0              1         1.0     0.0   
4234     1   51        3.0              1        43.0     0.0   
4237     0   52        2.0              0         0.0     0.0   

      prevalentStroke  prevalentHyp  diabetes  totChol  sysBP  diaBP    BMI  \
0                   0             0         0    195.0  106.0   70.0  26.97   
1                   0          

## 2. (4 points)

Using `age`, `currentSmoker`, `totChol`, `BMI`, and `heartRate` as the predictor variables, and `TenYearCHD` is the target variable. Define the 5-fold cross-validation strategy.

In [3]:
# Your code here
from sklearn.model_selection import StratifiedKFold

X = heart[["age", "currentSmoker", "totChol", "BMI", "heartRate"]]
y = heart["TenYearCHD"]

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

## 3. (8 points)

Train and evaluate an `ExtraTreesClassifier` model (with `n_estimators=500`, and `max_depth=3`) over the five-fold cross-validation strategy defined in part 2. Using 10% as a cutoff value, report the average recall over the cross-validation strategy.

In [4]:
# Your code here
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import recall_score
import numpy as np

et_md = ExtraTreesClassifier(n_estimators=500, 
                             max_depth=3,
                             random_state=42)

et_recalls = []
for i, (train_index, test_index) in enumerate(skf.split(X, y)):
    
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]
    
    et_md.fit(X_train, y_train)
    y_pred = et_md.predict_proba(X_test)[:, 1]
    y_pred = (y_pred >= 0.1).astype(int)
    
    score = recall_score(y_test, y_pred)
    et_recalls.append(score)
    print(f"Fold {i+1} Recall: {score}")
    
print(f"Mean Recall: {np.mean(et_recalls)}")

Fold 1 Recall: 0.9821428571428571
Fold 2 Recall: 0.9819819819819819
Fold 3 Recall: 1.0
Fold 4 Recall: 1.0
Fold 5 Recall: 0.9732142857142857
Mean Recall: 0.9874678249678249


## 4. (8 points)

Train and evaluate a `GradientBoostingClassifier` model (with `n_estimators=500` and `max_depth=3`, and `learning_rate=0.01`) over the five-fold cross-validation strategy defined in part 2. Using 10% as a cutoff value, report the average recall over the cross-validation strategy.

In [5]:
# Your code here
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import recall_score
import numpy as np

gb_md = GradientBoostingClassifier(n_estimators=500, 
                                   max_depth=3,
                                   learning_rate=0.01,
                                   random_state=42)

gb_recalls = []
for i, (train_index, test_index) in enumerate(skf.split(X, y)):
    
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]
    
    gb_md.fit(X_train, y_train)
    y_pred = gb_md.predict_proba(X_test)[:, 1]
    y_pred = (y_pred >= 0.1).astype(int)
    
    score = recall_score(y_test, y_pred)
    gb_recalls.append(score)
    print(f"Fold {i+1} Recall: {score}")
    
print(f"Mean Recall: {np.mean(gb_recalls)}")

Fold 1 Recall: 0.7053571428571429
Fold 2 Recall: 0.8468468468468469
Fold 3 Recall: 0.918918918918919
Fold 4 Recall: 0.8558558558558559
Fold 5 Recall: 0.8303571428571429
Mean Recall: 0.8314671814671815


## 5. (8 points)

Train and evaluate a `HistGradientBoostingClassifier` model (with `max_iter=500` and `max_depth=3`, and `learning_rate=0.01`) over the five-fold cross-validation strategy defined in part 2. Using 10% as a cutoff value, report the average recall over the cross-validation strategy.

In [6]:
# Your code here
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import recall_score
import numpy as np

hb_md = HistGradientBoostingClassifier(max_iter=500, 
                                       max_depth=3,
                                       learning_rate=0.01,
                                       random_state=42)

hb_recalls = []
for i, (train_index, test_index) in enumerate(skf.split(X, y)):
    
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]
    
    hb_md.fit(X_train, y_train)
    y_pred = hb_md.predict_proba(X_test)[:, 1]
    y_pred = (y_pred >= 0.1).astype(int)
    
    score = recall_score(y_test, y_pred)
    hb_recalls.append(score)
    print(f"Fold {i+1} Recall: {score}")
    
print(f"Mean Recall: {np.mean(hb_recalls)}")

Fold 1 Recall: 0.75
Fold 2 Recall: 0.8468468468468469
Fold 3 Recall: 0.9279279279279279
Fold 4 Recall: 0.8918918918918919
Fold 5 Recall: 0.8303571428571429
Mean Recall: 0.849404761904762


## 6. (8 points)

Train and evaluate a `LGBMClassifier` model (with `max_iter=500` and `max_depth=3`, and `learning_rate=0.01`) over the five-fold cross-validation strategy defined in part 2. Using 10% as a cutoff value, report the average recall over the cross-validation strategy.

In [7]:
# Your code here
from lightgbm import LGBMClassifier
from sklearn.metrics import recall_score
import numpy as np

lgb_md = LGBMClassifier(n_estimators=500,
                        max_depth=3,
                        learning_rate=0.01,
                        verbose=-1,
                        random_state=42)

lgb_recalls = []
for i, (train_index, test_index) in enumerate(skf.split(X, y)):
    
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]
    
    lgb_md.fit(X_train, y_train)
    y_pred = lgb_md.predict_proba(X_test)[:, 1]
    y_pred = (y_pred >= 0.1).astype(int)
    
    score = recall_score(y_test, y_pred)
    lgb_recalls.append(score)
    print(f"Fold {i+1} Recall: {score}")
    
print(f"Mean Recall: {np.mean(lgb_recalls)}")


Fold 1 Recall: 0.7410714285714286
Fold 2 Recall: 0.8468468468468469
Fold 3 Recall: 0.9279279279279279
Fold 4 Recall: 0.8918918918918919
Fold 5 Recall: 0.8303571428571429
Mean Recall: 0.8476190476190476


## 7. (8 points)

Using the models from parts 3 to 6 as bases estimators, train and evaluate a `StackingClassifier` over the five-fold cross-validation strategy defined in part 2. Using 10% as a cutoff value, report the average recall over the cross-validation strategy.

In [8]:
# Your code here
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import recall_score
from sklearn.ensemble import StackingClassifier
import numpy as np

logistic_md = LogisticRegression(max_iter=1000,
                                 random_state=42)

stacking_md = StackingClassifier(
    estimators=[('et', et_md),
                ('gb', gb_md),
                ('hb', hb_md),
                ('lgb', lgb_md)],
    final_estimator=logistic_md,
    cv=5,
    n_jobs=-1
)

stacking_recalls = []
for i, (train_index, test_index) in enumerate(skf.split(X, y)):
    
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]
    
    stacking_md.fit(X_train, y_train)
    y_pred = stacking_md.predict_proba(X_test)[:, 1]
    y_pred = (y_pred > 0.1).astype(int)
    
    score = recall_score(y_test, y_pred)
    stacking_recalls.append(score)
    print(f"Fold {i+1} Recall Score:", score)

print("Mean Recall Score:", np.mean(stacking_recalls))

Fold 1 Recall Score: 0.7857142857142857
Fold 2 Recall Score: 0.8828828828828829
Fold 3 Recall Score: 0.972972972972973
Fold 4 Recall Score: 0.918918918918919
Fold 5 Recall Score: 0.8660714285714286
Mean Recall Score: 0.8853120978120979


## 8. (3 points)

Using the results from parts 3-7, what model would you use to predict `TenYearCHD` based on the recall?

Based on the results I would yse the ExtraTreesClassifier model to predict TenYearCHD because its average recall is higher.